<a href="https://colab.research.google.com/github/Chiragdas21/Chiragdas21/blob/main/energy_forecasting_benchmark_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Short-Term Household Load Forecasting Benchmark
UCI Individual Household Electric Power Consumption, one-step-ahead (next hour).

Run the cells top to bottom. Results are saved in `./results`.

## 1. Install and download data

In [ ]:
!pip -q install xgboost statsmodels scikit-learn scipy torch

In [ ]:
!wget -q -O household.zip "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
!unzip -o household.zip
!head -3 household_power_consumption.txt

Archive:  household.zip
  inflating: household_power_consumption.txt  
Date;Time;Global_active_power;Global_reactive_power;Voltage;Global_intensity;Sub_metering_1;Sub_metering_2;Sub_metering_3
16/12/2006;17:24:00;4.216;0.418;234.840;18.400;0.000;1.000;17.000
16/12/2006;17:25:00;5.360;0.436;233.630;23.000;0.000;1.000;16.000


## 2. Code (config, data, metrics, models, main)

In [ ]:
import os, warnings, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

In [ ]:
# ------------------------- CONFIG -------------------------
DATA_PATH = "household_power_consumption.txt"
OUT = "results"
SEEDS = [0, 1, 2, 3, 4]
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15          # remaining 0.15 = test
LAGS = [1, 2, 3, 24, 48, 168]
ROLL_WINDOWS = [3, 24, 168]
RUN_SARIMA = True      # slow-ish; fits on last 2000 train hours
RUN_LSTM = True        # needs torch
LSTM_WINDOW = 48
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ------------------------- DATA -------------------------
def load_hourly(path):
    df = pd.read_csv(path, sep=";", na_values=["?"], low_memory=False)
    df["dt"] = pd.to_datetime(df["Date"] + " " + df["Time"],
                              format="%d/%m/%Y %H:%M:%S")
    df = df.set_index("dt").sort_index()
    p = pd.to_numeric(df["Global_active_power"], errors="coerce")  # kW, per minute
    n_rows, n_missing = len(p), int(p.isna().sum())
    # mean kW over an hour == kWh consumed in that hour
    hourly = p.resample("h").mean()
    n_hours_missing = int(hourly.isna().sum())
    hourly = hourly.interpolate(method="time", limit=6, limit_area="inside")
    hourly = hourly.dropna()
    stats_ = dict(minute_rows=n_rows, minute_missing=n_missing,
                  minute_missing_pct=round(100 * n_missing / n_rows, 3),
                  hours_missing_before_interp=n_hours_missing,
                  hours_final=len(hourly),
                  start=str(hourly.index.min()), end=str(hourly.index.max()))
    json.dump(stats_, open(f"{OUT}/dataset_stats.json", "w"), indent=2)
    print("Dataset stats:", stats_)
    return hourly.rename("kwh")


def make_features(s):
    d = pd.DataFrame({"y": s})
    for l in LAGS:
        d[f"lag_{l}"] = s.shift(l)
    for w in ROLL_WINDOWS:   # shifted by 1 -> no leakage of current target
        d[f"rollmean_{w}"] = s.shift(1).rolling(w).mean()
    d["rollstd_24"] = s.shift(1).rolling(24).std()
    idx = d.index
    d["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
    d["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    d["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    d["month_sin"] = np.sin(2 * np.pi * idx.month / 12)
    d["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    d["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    return d.dropna()

In [ ]:
# ------------------------- METRICS -------------------------
def smape(y, p):
    return 100 * np.mean(2 * np.abs(p - y) / (np.abs(y) + np.abs(p) + 1e-8))


def metrics(y, p):
    return dict(MAE=mean_absolute_error(y, p),
                RMSE=np.sqrt(mean_squared_error(y, p)),
                sMAPE=smape(y, p), R2=r2_score(y, p))


def dm_test(y, p1, p2, loss="se"):
    """Diebold-Mariano (h=1). H0: equal accuracy. Negative stat => model 1 better."""
    e1, e2 = y - p1, y - p2
    d = (e1 ** 2 - e2 ** 2) if loss == "se" else (np.abs(e1) - np.abs(e2))
    n = len(d)
    # Newey-West variance, lag 24
    dm = d - d.mean()
    var = np.sum(dm ** 2) / n
    for k in range(1, 25):
        w = 1 - k / 25
        var += 2 * w * np.sum(dm[k:] * dm[:-k]) / n
    stat = d.mean() / np.sqrt(var / n)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))

In [ ]:
# ------------------------- MODELS -------------------------
def get_xgb(seed, **kw):
    try:
        from xgboost import XGBRegressor
        return XGBRegressor(random_state=seed, n_jobs=-1, **kw)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        print("xgboost missing -> using HistGradientBoosting")
        return HistGradientBoostingRegressor(random_state=seed)


def tune(make_model, grid, Xtr, ytr, Xva, yva):
    best, best_rmse = None, np.inf
    for params in grid:
        m = make_model(0, **params).fit(Xtr, ytr)
        r = np.sqrt(mean_squared_error(yva, m.predict(Xva)))
        if r < best_rmse:
            best, best_rmse = params, r
    print("  best params:", best, "val RMSE: %.4f" % best_rmse)
    return best


def run_lstm(s, tr_end, va_end, test_idx, seed):
    import torch, torch.nn as nn
    torch.manual_seed(seed); np.random.seed(seed)
    mu, sd = s.iloc[:tr_end].mean(), s.iloc[:tr_end].std()
    z = ((s - mu) / sd).values.astype(np.float32)
    idx = s.index
    cal = np.stack([np.sin(2*np.pi*idx.hour/24), np.cos(2*np.pi*idx.hour/24),
                    np.sin(2*np.pi*idx.dayofweek/7), np.cos(2*np.pi*idx.dayofweek/7)],
                   1).astype(np.float32)
    feats = np.concatenate([z[:, None], cal], 1)
    W = LSTM_WINDOW

    def build(lo, hi):
        X = np.stack([feats[t - W:t] for t in range(lo, hi)])
        return torch.tensor(X), torch.tensor(z[lo:hi])
    Xtr, ytr = build(W, tr_end)
    Xva, yva = build(tr_end, va_end)
    Xte, yte = build(va_end, len(s))

    class Net(nn.Module):
        def __init__(s_):
            super().__init__()
            s_.l = nn.LSTM(5, 64, batch_first=True); s_.o = nn.Linear(64, 1)
        def forward(s_, x):
            h, _ = s_.l(x); return s_.o(h[:, -1]).squeeze(-1)
    net = Net(); opt = torch.optim.Adam(net.parameters(), 1e-3); lf = nn.MSELoss()
    best, bad, best_state = np.inf, 0, None
    for ep in range(40):
        net.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            opt.zero_grad(); lf(net(Xtr[b]), ytr[b]).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            v = lf(net(Xva), yva).item()
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        pred = net(Xte).numpy() * sd + mu
    # align to feature-frame test index
    out = pd.Series(pred, index=s.index[va_end:])
    return out.reindex(test_idx).values


def run_sarima(s, tr_end, va_end, test_idx):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    tail = s.iloc[max(0, tr_end - 2000):tr_end]
    res = SARIMAX(tail, order=(1, 0, 1), seasonal_order=(1, 1, 0, 24),
                  enforce_stationarity=False, enforce_invertibility=False
                  ).fit(disp=False, maxiter=50)
    full = s.iloc[max(0, tr_end - 2000):]
    applied = res.apply(full)                       # same params, true history
    pred = applied.get_prediction(start=va_end - (tr_end - 2000)).predicted_mean
    return pred.reindex(test_idx).values            # one-step-ahead

In [ ]:
# ------------------------- MAIN -------------------------
def main():
    s = load_hourly(DATA_PATH)
    F = make_features(s)
    n = len(F)
    tr_end, va_end = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    tr, va, te = F.iloc[:tr_end], F.iloc[tr_end:va_end], F.iloc[va_end:]
    feat_cols = [c for c in F.columns if c != "y"]
    Xtr, ytr = tr[feat_cols], tr["y"]
    Xva, yva = va[feat_cols], va["y"]
    Xte, yte = te[feat_cols], te["y"].values
    print(f"Split: train {tr.index[0]}..{tr.index[-1]} ({len(tr)}), "
          f"val ({len(va)}), test {te.index[0]}..{te.index[-1]} ({len(te)})")

    preds, rows = {}, []

    def log(name, p, seed_metrics=None):
        m = metrics(yte, p)
        row = {"Model": name, **m}
        if seed_metrics:
            for k in ["MAE", "RMSE", "sMAPE", "R2"]:
                v = [x[k] for x in seed_metrics]
                row[k] = np.mean(v); row[k + "_std"] = np.std(v)
        rows.append(row); preds[name] = p
        print(f"{name:22s} MAE {row['MAE']:.4f} RMSE {row['RMSE']:.4f} R2 {row['R2']:.4f}")

    # --- naive baselines (all use only past data) ---
    log("Persistence (t-1)", te["lag_1"].values)
    log("Seasonal naive (24h)", te["lag_24"].values)
    log("Seasonal naive (168h)", te["lag_168"].values)

    # --- Ridge / Linear ---
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.linear_model import LinearRegression
    lr = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
    log("Linear Regression", lr.predict(Xte))
    rd = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xtr, ytr)
    log("Ridge", rd.predict(Xte))

    # --- Random Forest (tuned on validation) ---
    print("Tuning RF...")
    rf_grid = [dict(n_estimators=n_, max_depth=d_, min_samples_leaf=l_)
               for n_ in (200, 400) for d_ in (10, 20, None) for l_ in (1, 5)]
    rf_make = lambda seed, **kw: RandomForestRegressor(random_state=seed, n_jobs=-1, **kw)
    rf_best = tune(rf_make, rf_grid, Xtr, ytr, Xva, yva)
    Xfit = pd.concat([Xtr, Xva]); yfit = pd.concat([ytr, yva])  # refit on train+val
    sm, rf_models = [], []
    for sd in SEEDS:
        m = rf_make(sd, **rf_best).fit(Xfit, yfit); rf_models.append(m)
        sm.append(metrics(yte, m.predict(Xte)))
    log("Random Forest", rf_models[0].predict(Xte), sm)

    # --- XGBoost (tuned on validation) ---
    print("Tuning XGBoost...")
    xgb_grid = [dict(n_estimators=n_, max_depth=d_, learning_rate=lr_, subsample=0.8)
                for n_ in (300, 600) for d_ in (4, 6) for lr_ in (0.03, 0.1)]
    xgb_best = tune(get_xgb, xgb_grid, Xtr, ytr, Xva, yva)
    json.dump({"random_forest": rf_best, "xgboost": xgb_best},
              open(f"{OUT}/best_params.json", "w"), indent=2, default=str)
    sm, xg_models = [], []
    for sd in SEEDS:
        m = get_xgb(sd, **xgb_best).fit(Xfit, yfit); xg_models.append(m)
        sm.append(metrics(yte, m.predict(Xte)))
    log("XGBoost", xg_models[0].predict(Xte), sm)

    # --- SARIMA ---
    if RUN_SARIMA:
        try:
            print("Fitting SARIMA...")
            p = run_sarima(s, tr_end + 0, va_end, te.index)
            ok = ~np.isnan(p)
            if ok.mean() > 0.99:
                log("SARIMA(1,0,1)(1,1,0)24", np.where(ok, p, te["lag_1"].values))
        except Exception as e:
            print("SARIMA skipped:", e)

    # --- LSTM ---
    if RUN_LSTM:
        try:
            print("Training LSTM...")
            # positions in the raw series s (not F) -> map via timestamps
            pos_tr = s.index.get_loc(tr.index[-1]) + 1
            pos_va = s.index.get_loc(va.index[-1]) + 1
            sm, first = [], None
            for sd in SEEDS[:3]:
                p = run_lstm(s, pos_tr, pos_va, te.index, sd)
                sm.append(metrics(yte, p)); first = p if first is None else first
            log("LSTM", first, sm)
        except Exception as e:
            print("LSTM skipped:", e)

    # --- save main table ---
    res = pd.DataFrame(rows).set_index("Model")
    res.to_csv(f"{OUT}/metrics_test.csv")
    with open(f"{OUT}/metrics_table.tex", "w") as f:
        f.write(res[["MAE", "RMSE", "sMAPE", "R2"]].round(4).to_latex())
    print("\n", res.round(4))

    # --- Diebold-Mariano vs best model (by RMSE) ---
    best_name = res["RMSE"].idxmin()
    dm_rows = []
    for name, p in preds.items():
        if name == best_name: continue
        st, pv = dm_test(yte, preds[best_name], p)
        dm_rows.append(dict(best=best_name, other=name, DM_stat=st, p_value=pv))
    pd.DataFrame(dm_rows).to_csv(f"{OUT}/dm_tests.csv", index=False)

    # --- walk-forward CV (5 expanding-window folds, over train+val) ---
    wf = []
    tscv = TimeSeriesSplit(n_splits=5, test_size=len(Xfit) // 8)
    for k, (a, b) in enumerate(tscv.split(Xfit)):
        Xa, ya, Xb, yb = Xfit.iloc[a], yfit.iloc[a], Xfit.iloc[b], yfit.iloc[b]
        cand = {"Persistence": Xb["lag_1"].values,
                "Seasonal naive 24h": Xb["lag_24"].values,
                "Ridge": make_pipeline(StandardScaler(), Ridge()).fit(Xa, ya).predict(Xb),
                "Random Forest": rf_make(0, **rf_best).fit(Xa, ya).predict(Xb),
                "XGBoost": get_xgb(0, **xgb_best).fit(Xa, ya).predict(Xb)}
        for name, p in cand.items():
            wf.append(dict(fold=k, model=name, **metrics(yb.values, p)))
    wf = pd.DataFrame(wf)
    wf.to_csv(f"{OUT}/walkforward.csv", index=False)
    print("\nWalk-forward (mean over folds):\n",
          wf.groupby("model")[["MAE", "RMSE", "R2"]].agg(["mean", "std"]).round(4))

    # --- feature importance (RF + XGB) ---
    fi = pd.DataFrame({"feature": feat_cols,
                       "RF": rf_models[0].feature_importances_,
                       "XGB": getattr(xg_models[0], "feature_importances_",
                                      np.zeros(len(feat_cols)))})
    fi.sort_values("RF", ascending=False).to_csv(f"{OUT}/feature_importance.csv", index=False)

    # --- figures ---
    best_p = preds[best_name]
    err = yte - best_p
    # 1) actual vs predicted, last 168h of test (fixed rule, not cherry-picked)
    plt.figure(figsize=(9, 3.2))
    plt.plot(te.index[-168:], yte[-168:], label="Actual")
    plt.plot(te.index[-168:], best_p[-168:], "--", label=f"Predicted ({best_name})")
    plt.ylabel("kWh"); plt.legend(); plt.title("Last 7 days of test set")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_actual_vs_pred.png", dpi=200); plt.close()
    # 2) error by hour of day / weekday
    e = pd.DataFrame({"abs_err": np.abs(err), "hour": te.index.hour,
                      "dow": te.index.dayofweek})
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    e.groupby("hour")["abs_err"].mean().plot.bar(ax=ax[0], title="MAE by hour")
    e.groupby("dow")["abs_err"].mean().plot.bar(ax=ax[1], title="MAE by weekday (0=Mon)")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_error_by_time.png", dpi=200); plt.close()
    # 3) error histogram
    plt.figure(figsize=(5, 3)); plt.hist(err, bins=40, edgecolor="k")
    plt.xlabel("Error (kWh)"); plt.title("Prediction error distribution")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_error_hist.png", dpi=200); plt.close()
    # 4) peak-hour error (top 10% demand hours)
    thr = np.quantile(yte, 0.9)
    pk = yte >= thr
    peak = dict(threshold_kwh=float(thr),
                MAE_peak=float(np.mean(np.abs(err[pk]))),
                MAE_offpeak=float(np.mean(np.abs(err[~pk]))))
    json.dump(peak, open(f"{OUT}/peak_error.json", "w"), indent=2)
    print("Peak-hour error:", peak)
    # --- save test predictions (for any later analysis) ---
    pd.DataFrame(preds, index=te.index).assign(actual=yte).to_csv(
        f"{OUT}/test_predictions.csv")

    # --- peak-hour detection as a classification view (confusion matrix) ---
    # "peak" = actual demand in the top 10% of test hours (threshold thr above);
    # a model "flags" a peak when its predicted demand is >= thr.
    from sklearn.metrics import (precision_score, recall_score, f1_score,
                                 confusion_matrix)
    y_pk = (yte >= thr).astype(int)
    pk_rows = []
    for name, p in preds.items():
        flag = (np.asarray(p) >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_pk, flag, labels=[0, 1]).ravel()
        pk_rows.append(dict(Model=name, TP=tp, FP=fp, FN=fn, TN=tn,
                            Precision=precision_score(y_pk, flag, zero_division=0),
                            Recall=recall_score(y_pk, flag, zero_division=0),
                            F1=f1_score(y_pk, flag, zero_division=0)))
    pk_df = pd.DataFrame(pk_rows)
    pk_df.to_csv(f"{OUT}/peak_detection.csv", index=False)
    print("\nPeak-hour detection (threshold %.3f kWh):\n" % thr,
          pk_df.round(3).to_string(index=False))
    cm = confusion_matrix(y_pk, (best_p >= thr).astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=12)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred normal", "Pred peak"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual normal", "Actual peak"])
    ax.set_title(f"Peak detection: {best_name}", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_peak_confusion.png", dpi=300); plt.close()

    # 5) feature importance
    fi.sort_values("RF").tail(12).plot.barh(x="feature", y="RF", figsize=(5, 4), legend=False)
    plt.title("RF feature importance"); plt.tight_layout()
    plt.savefig(f"{OUT}/fig_feature_importance.png", dpi=200); plt.close()
    # 6) model comparison
    res[["MAE", "RMSE"]].plot.bar(figsize=(9, 3.5), title="Test error by model")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_model_comparison.png", dpi=200); plt.close()
    print(f"\nDone. Everything saved in ./{OUT}/")

## 3. Run the benchmark

In [ ]:
main()

Dataset stats: {'minute_rows': 2075259, 'minute_missing': 25979, 'minute_missing_pct': 1.252, 'hours_missing_before_interp': 421, 'hours_final': 34211, 'start': '2006-12-16 17:00:00', 'end': '2010-11-26 21:00:00'}
Split: train 2006-12-23 17:00:00..2009-09-16 05:00:00 (23830), val (5106), test 2010-04-20 01:00:00..2010-11-26 21:00:00 (5107)
Persistence (t-1)      MAE 0.3851 RMSE 0.5837 R2 0.3224
Seasonal naive (24h)   MAE 0.5132 RMSE 0.7613 R2 -0.1526
Seasonal naive (168h)  MAE 0.5627 RMSE 0.8091 R2 -0.3021
Linear Regression      MAE 0.3536 RMSE 0.5009 R2 0.5010
Ridge                  MAE 0.3536 RMSE 0.5009 R2 0.5010
Tuning RF...
  best params: {'n_estimators': 400, 'max_depth': 20, 'min_samples_leaf': 5} val RMSE: 0.5397
Random Forest          MAE 0.3238 RMSE 0.4679 R2 0.5646
Tuning XGBoost...
  best params: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.8} val RMSE: 0.5366
XGBoost                MAE 0.3220 RMSE 0.4638 R2 0.5721
Fitting SARIMA...


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, fr

SARIMA(1,0,1)(1,1,0)24 MAE 0.3997 RMSE 0.5694 R2 0.3552
Training LSTM...
LSTM                   MAE 0.3373 RMSE 0.4815 R2 0.5389

                            MAE    RMSE    sMAPE      R2  MAE_std  RMSE_std  \
Model                                                                        
Persistence (t-1)       0.3851  0.5837  38.5941  0.3224      NaN       NaN   
Seasonal naive (24h)    0.5132  0.7613  49.4740 -0.1526      NaN       NaN   
Seasonal naive (168h)   0.5627  0.8091  55.6400 -0.3021      NaN       NaN   
Linear Regression       0.3536  0.5009  38.2226  0.5010      NaN       NaN   
Ridge                   0.3536  0.5009  38.2228  0.5010      NaN       NaN   
Random Forest           0.3238  0.4679  34.7461  0.5646   0.0002    0.0005   
XGBoost                 0.3220  0.4638  34.7759  0.5721   0.0004    0.0005   
SARIMA(1,0,1)(1,1,0)24  0.3997  0.5694  43.5883  0.3552      NaN       NaN   
LSTM                    0.3373  0.4815  36.0815  0.5389   0.0028    0.0032   

          

## 4. Download results

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn, xgboost, statsmodels, torch
print(sklearn.__version__, xgboost.__version__, statsmodels.__version__, torch.__version__)

1.6.1 3.4.1 0.15.0 2.14.1+cu130
